# 函数类型与重载

学习目标：能描述函数的调用方式、回调与重载，并区分静态签名和实际调用规则。

前置知识：JavaScript 函数、回调、this、参数默认值与展开；TypeScript 联合、元组和收窄。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ES 模块，开启 strict。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/07-functions-and-overloads/。

1. [main.ts](scripts/07-functions-and-overloads/main.ts)：按正文顺序组织的正常示例，片段依赖同文件前文定义。
2. [type-errors.ts](scripts/07-functions-and-overloads/type-errors.ts)：与正常示例隔离的类型反例，不生成或执行 JavaScript。
3. [tsconfig.json](scripts/07-functions-and-overloads/tsconfig.json)、[tsconfig.errors.json](scripts/07-functions-and-overloads/tsconfig.errors.json)：分别明确正常与反例文件范围。



Step 1：检查正常项目的类型。

```bash
npm run check:07
```

Step 2：生成正常项目的 JavaScript。

```bash
npm run build:07
```

Step 3：运行正常示例。

```bash
npm run run:07
```

Step 4：检查下文独立列出的类型反例。

```bash
npm run errors:07
# 预期非零退出；按反例注释逐行核对具体错误，不运行 type-errors.ts。
```

正常配置只包含上面列出的正常与独立运行示例，生成文件位于 .build/07-functions-and-overloads/。错误配置继承正常选项，改用 type-errors.ts 并开启 noEmit。

## 1 函数类型表达式与返回值

函数也是值，可以写成 (value: number) =&gt; string。value 是类型签名中的参数名，number 是该参数类型，箭头后的 string 是返回类型。名称不需要与实现形参同名，但不能省略名称而误写成 (number) =&gt; string。

上下文类型能为回调参数提供类型。返回值可以推断，也可显式标注来固定对外契约；标注不会转换实际返回值。异步函数的 Promise 返回类型在泛型章节展开。

```typescript
export {};
type Formatter = (value: number) => string;
const format: Formatter = (amount) => amount.toFixed(1);
function apply(value: number, formatter: Formatter): string { return formatter(value); }
console.log(apply(2, format)); // 2.0
```

以下片段来自独立的 type-errors.ts：

```typescript
const wrongReturn: (value: number) => string = (value) => value; // 返回 number，不能满足 string 契约。
```

## 2 调用签名与构造签名

函数值还可以拥有属性，此时使用对象类型中的调用签名；参数列表后用冒号写返回类型。构造签名在前面加 new，表示可以用 new 调用并产生指定实例的值，它与普通调用签名不同。

下面 Searcher 同时需要函数能力和 label 属性。Maker 只描述构造侧，生成的对象只需有 title；类的静态侧与实例侧将在类章节进一步区分。

```typescript
type Searcher = { label: string; (query: string): boolean };
function containsType(query: string): boolean { return query.includes("类型"); }
containsType.label = "检索";
const searcher: Searcher = containsType;
type Maker = new (title: string) => { title: string };
class Note {
  title: string;
  constructor(title: string) { this.title = title; }
}
function makeNote(creator: Maker): { title: string } { return new creator("函数"); }
console.log(searcher.label, searcher("类型系统"), makeNote(Note).title); // 检索 true 函数
```

以下片段来自独立的 type-errors.ts：

```typescript
type NeedsConstructor = new () => { title: string };
const plainFunction: NeedsConstructor = () => ({ title: "函数" }); // 箭头函数没有构造签名。
```

## 3 可选参数与默认参数

可选参数可省略，也可显式传 undefined；函数体内读取时需考虑 undefined。默认参数在省略或传 undefined 时取默认值，不会把 null 当成缺省值。

可选参数通常位于必需参数之后。默认参数若位于必需参数之前，调用者仍要提供该位置，必要时传 undefined 来使用默认值。exactOptionalPropertyTypes 针对对象可选属性，不改变这里可选参数允许 undefined 的规则。

```typescript
function greeting(name: string, suffix?: string): string { return name + (suffix ?? "!"); }
function repeat(text: string, count = 2): string { return text.repeat(count); }
function tagged(prefix = "课", title: string): string { return prefix + title; }
console.log(greeting("林"), greeting("林", undefined), repeat("A", undefined), tagged(undefined, "函数")); // 林! 林! AA 课函数
```

以下片段来自独立的 type-errors.ts：

```typescript
function needsSecond(prefix = "课", title: string): string { return prefix + title; }
needsSecond("函数"); // 后面的 title 仍是必需参数。
```

## 4 剩余参数、元组展开与回调

剩余参数把其余实参收集为数组，也可用元组表达位置不同的参数。要把数组展开给固定参数函数，编译器需知道其长度和位置；使用元组标注或 as const 可保留这类信息。

回调参数写 ? 表示调用者可能不提供该实参，不能仅因为回调实现可能忽略它就写 ?。通常允许只声明用得上的回调形参。下面调用方保证提供索引，而回调可以只接收值。

```typescript
function total(...values: number[]): number { return values.reduce((sum, value) => sum + value, 0); }
function label(...args: [title: string, minutes: number]): string { return args[0] + ":" + args[1]; }
const args: [string, number] = ["函数", 18];
function visit(callback: (value: string, index: number) => void): void { callback("TS", 0); }
const seen: string[] = [];
visit((value) => { seen.push(value); });
console.log(total(1, 2, 3), label(...args), seen.join(",")); // 6 函数:18 TS
```

以下片段来自独立的 type-errors.ts：

```typescript
function fixedCall(title: string, minutes: number): void {}
const loose = ["函数", 18];
fixedCall(...loose); // 普通联合元素数组没有确定的两个位置。
function optionalIndex(callback: (value: string, index?: number) => void): void { callback("TS"); }
optionalIndex((value, index) => { index.toFixed(); }); // index 可能未提供。
```

## 5 重载签名与实现签名

重载（overload）为同一个函数提供多个对外调用签名，最后保留一个兼容这些签名的实现。实现签名对外不可直接调用；即使它接受联合或可选参数，调用仍需匹配公开重载。

本例用字符串输入得到数值、数值输入得到字符串，重载保留输入与输出的对应关系。实现必须真实检查输入。若只是同一返回类型、同一参数个数的多种输入，通常一个联合参数更容易使用，也能直接接收联合实参。

```typescript
function convert(value: string): number;
function convert(value: number): string;
function convert(value: string | number): number | string {
  return typeof value === "string" ? value.length : value.toFixed(0);
}
const length: number = convert("函数");
const digits: string = convert(12);
function size(value: string | readonly unknown[]): number { return value.length; }
console.log(length, digits, size([1, 2, 3])); // 2 12 3
```

以下片段来自独立的 type-errors.ts：

```typescript
function pick(value: string): number;
function pick(value: number): string;
function pick(value: string | number): string | number { return typeof value === "string" ? value.length : value.toFixed(0); }
function callUnion(value: string | number): void { pick(value); } // 联合实参不匹配任一个公开重载。
function onlyOne(value: string): string;
function onlyOne(value?: string): string { return value ?? "默认"; }
onlyOne(); // 实现虽有可选参数，但公开重载要求一个参数。
```

## 6 this 参数与 void 回调

函数签名最前面的 this 参数描述调用时的接收者，它在输出中被擦除，不占一个实际实参位置。使用 call 提供接收者；这个类型声明不会自动绑定 this。箭头函数捕获外层 this，不适合用作需要动态接收者的实现。

返回 void 的回调契约表示调用方不使用其返回值，因此可以接收返回其他值的函数。若函数定义自身显式标注 : void，则不能返回普通数值。两者区别在于“忽略回调结果”和“实现声明的返回契约”。

```typescript
function titleWithPrefix(this: { prefix: string }, title: string): string {
  return this.prefix + title;
}
const storage: number[] = [];
const sink: (value: number) => void = (value) => storage.push(value);
sink(3); // 返回值在这个静态视图下不可作为 number 使用。
console.log(titleWithPrefix.call({ prefix: "课程:" }, "函数"), storage.length); // 课程:函数 1
```

以下片段来自独立的 type-errors.ts：

```typescript
function needsThis(this: { prefix: string }, title: string): string { return this.prefix + title; }
needsThis("函数"); // 普通调用没有满足要求的 this 接收者。
function explicitVoid(): void { return 1; } // 显式 void 实现不能返回 number。
const discard: () => void = () => 1;
const result: number = discard(); // 调用方看到 void，不能当作 number。
```

## 本章小结

签名描述调用要求，返回标注不会转换值。重载的实现必须覆盖公开签名；元组展开保留参数位置。this 参数是静态接收者约束，void 回调允许忽略返回值。

## 练习

1. 实现接受一个标题或标题与时长的函数，使用可选参数或元组联合；检查正常调用，并让多余的第三个参数产生诊断。
2. 为字符串与数值输入设计返回不同类型的重载，给两种结果分别赋给对应类型变量，确认静态检查和实际行为一致。
3. 将需要 this 的函数通过 bind 固定接收者后调用；结果与 call 一致。另说明把返回数值的函数赋给 void 回调后为何不能使用静态返回值。

## 参考与引用来源

- TypeScript 官方文档：[More on Functions：调用、构造、可选参数、重载、this 与 void](https://www.typescriptlang.org/docs/handbook/2/functions.html)；[Everyday Types：参数、返回值与上下文类型](https://www.typescriptlang.org/docs/handbook/2/everyday-types.html#functions)；[Type Compatibility：丢弃参数](https://www.typescriptlang.org/docs/handbook/type-compatibility.html#comparing-two-functions)；[4.0：具名元组参数](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-4-0.html#labeled-tuple-elements)。

- npm 官方文档：[npm run（v11）](https://docs.npmjs.com/cli/v11/commands/npm-run/)：从本技术目录运行已配置脚本，并解析本地工具。